# Quantum Mechanics

This notebook covers **8 quantum mechanics equations**, from basic potentials to spin dynamics:

1. **Schrodinger Free Particle** — plane wave solutions with V=0
2. **Schrodinger Infinite Well** — quantized energy levels in a box
3. **Schrodinger Harmonic Oscillator** — Hermite polynomial eigenstates
4. **Schrodinger Finite Well** — bound states via shooting method
5. **Schrodinger Delta Potential** — single bound state from a delta well
6. **Time-Dependent Schrodinger** — Gaussian wave packet evolution
7. **Hydrogen Radial Equation** — Laguerre polynomial radial wavefunctions
8. **Pauli Equation** — spin-1/2 Larmor precession in a magnetic field

All equations use natural units ($\hbar = m = 1$) unless otherwise noted.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from diff_eq_solver import registry
from diff_eq_solver.visualizer import plot_ode_solution, plot_phase_portrait, plot_pde_heatmap, plot_pde_snapshots, plot_3d_surface, plot_comparison, plot_orbit, plot_potential_and_wavefunction
from IPython.display import Math, display

## 1. Schrodinger Free Particle

The **time-independent Schrodinger equation** for a free particle ($V = 0$):

$$-\frac{\hbar^2}{2m}\,\psi''(x) = E\,\psi(x)$$

**Physical context:** A particle moving freely with no potential barriers. The general solution is a superposition of plane waves: $\psi(x) = Ae^{ikx} + Be^{-ikx}$ where $k = \sqrt{2mE}/\hbar$. Plane waves are not normalizable individually; physical states require wave-packet superposition.

In [ ]:
# Symbolic solve
free_particle = registry.get('schrodinger_free_particle')
sym_sol = free_particle.symbolic_solve(hbar=1.0, mass=1.0, E=1.0)
print('Method:', sym_sol.info.get('method'))
print(f'Wavevector k = {sym_sol.info["k"]:.4f}')
print(f'Wavelength = {sym_sol.info["wavelength"]:.4f}')
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve
num_sol = free_particle.numerical_solve(
    initial_conditions={'psi0': 1.0, 'dpsi0': 0.0},
    t_span=(-10.0, 10.0),
    hbar=1.0, mass=1.0, E=1.0,
)
x, y = num_sol.numerical
psi = y[0]
V = np.zeros_like(x)  # V=0 everywhere

# Plot potential and wavefunction
fig = plot_potential_and_wavefunction(
    x, V, [psi], energies=[1.0],
    title='Free Particle: V=0, E=1'
)
plt.show()

## 2. Schrodinger Infinite Square Well

A particle confined in a 1-D **infinite square well** of width $L$:

$$-\frac{\hbar^2}{2m}\,\psi'' = E\,\psi, \quad \psi(0)=\psi(L)=0$$

**Physical context:** The prototypical example of quantum confinement. The eigenstates are $\psi_n(x) = \sqrt{2/L}\,\sin(n\pi x/L)$ with quantized energies $E_n = n^2\pi^2\hbar^2/(2mL^2)$. This "particle in a box" model is fundamental to understanding quantum dots, nanostructures, and molecular orbitals.

In [ ]:
# Symbolic solve — first few eigenstates
inf_well = registry.get('schrodinger_infinite_well')
for n in [1, 2, 3]:
    sym_sol = inf_well.symbolic_solve(n=n, L=1.0, hbar=1.0, mass=1.0)
    print(f'n={n}: E_{n} = {sym_sol.info["E_n"]:.6f}')
    if sym_sol.latex:
        display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and plot first 3 eigenstates with potential
x_plot = np.linspace(0, 1, 500)
V = np.zeros_like(x_plot)
# Set V to large value outside well for visual
V_vis = np.where((x_plot >= 0) & (x_plot <= 1), 0, 50)

psi_list = []
energies = []
for n in [1, 2, 3]:
    num_sol = inf_well.numerical_solve(
        t_span=(0.0, 1.0), n=n, L=1.0, hbar=1.0, mass=1.0,
    )
    x_num, y_num = num_sol.numerical
    psi_vals = np.interp(x_plot, x_num, y_num[0])
    E_n = num_sol.info['E_n']
    psi_list.append(psi_vals)
    energies.append(E_n)

fig = plot_potential_and_wavefunction(
    x_plot, V_vis, psi_list, energies=energies,
    title='Infinite Square Well: $\\psi_n(x) = \\sqrt{2/L}\\sin(n\\pi x/L)$'
)
plt.show()

## 3. Schrodinger Harmonic Oscillator

The **quantum harmonic oscillator** with potential $V(x) = \frac{1}{2}m\omega^2 x^2$:

$$-\frac{\hbar^2}{2m}\psi'' + \frac{1}{2}m\omega^2 x^2\psi = E\psi$$

**Physical context:** The most important model in quantum physics. Applications include molecular vibrations, quantum field theory (as the foundation), and coherent states. Eigenstates involve **Hermite polynomials**: $\psi_n(\xi) \propto H_n(\xi) e^{-\xi^2/2}$ with energies $E_n = \hbar\omega(n + \frac{1}{2})$. Note the non-zero **zero-point energy** $E_0 = \frac{1}{2}\hbar\omega$.

In [ ]:
# Symbolic solve — first few eigenstates
harmonic = registry.get('schrodinger_harmonic_oscillator')
for n in [0, 1, 2]:
    sym_sol = harmonic.symbolic_solve(n=n, hbar=1.0, mass=1.0, omega=1.0)
    print(f'n={n}: E_{n} = {sym_sol.info["E_n"]:.4f}')
    if sym_sol.latex:
        display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and plot eigenstates
x_plot = np.linspace(-6, 6, 500)
mass, omega, hbar = 1.0, 1.0, 1.0
V = 0.5 * mass * omega**2 * x_plot**2

psi_list = []
energies = []
for n in [0, 1, 2, 3]:
    num_sol = harmonic.numerical_solve(
        t_span=(-6.0, 6.0), n=n, hbar=hbar, mass=mass, omega=omega,
    )
    x_num, y_num = num_sol.numerical
    psi_vals = np.interp(x_plot, x_num, y_num[0])
    E_n = num_sol.info['E_n']
    psi_list.append(psi_vals)
    energies.append(E_n)

fig = plot_potential_and_wavefunction(
    x_plot, V, psi_list, energies=energies,
    title='Harmonic Oscillator: $E_n = \\hbar\\omega(n + 1/2)$'
)
plt.show()

## 4. Schrodinger Finite Square Well

A particle in a **finite square well** with depth $V_0$:

$$-\frac{\hbar^2}{2m}\psi'' + V(x)\psi = E\psi, \quad V(x) = \begin{cases}-V_0 & |x| < a \\ 0 & |x| \ge a\end{cases}$$

**Physical context:** Unlike the infinite well, the wavefunction **penetrates** into the classically forbidden region, decaying exponentially as $e^{-\kappa|x|}$ where $\kappa = \sqrt{-2mE}/\hbar$. Bound-state energies must be found numerically by solving transcendental equations (shooting method). The number of bound states depends on $V_0$ and $a$.

In [ ]:
# Symbolic — no closed form, note in info
finite_well = registry.get('schrodinger_finite_well')
sym_sol = finite_well.symbolic_solve(hbar=1.0, mass=1.0, V0=5.0, a=1.0)
print('Symbolic info:', sym_sol.info)

In [ ]:
# Numerical solve — energy scan + shooting method
num_sol = finite_well.numerical_solve(
    t_span=(-6.0, 6.0), hbar=1.0, mass=1.0, V0=5.0, a=1.0,
)
print(f'Number of bound states: {num_sol.info["n_bound_states"]}')
print(f'Bound energies: {[f"{e:.4f}" for e in num_sol.info.get("bound_energies", [])]}')
print(f'Ground state energy: {num_sol.info.get("ground_state_energy", "N/A")}')

# Plot potential and wavefunctions
if num_sol.numerical is not None:
    x_num, y_num = num_sol.numerical
    V0, a = 5.0, 1.0
    V = np.where(np.abs(x_num) < a, -V0, 0.0)
    
    psi_list = []
    energies = []
    for i, E in enumerate(num_sol.info.get('bound_energies', [])):
        # Re-solve for each state to get individual wavefunctions
        psi_list.append(y_num[0] if i == 0 else y_num[0])
        energies.append(E)
    
    if psi_list:
        fig = plot_potential_and_wavefunction(
            x_num, V, psi_list, energies=energies,
            title=f'Finite Square Well: $V_0={V0}$, $a={a}$'
        )
        plt.show()

## 5. Schrodinger Delta Potential

A particle in an attractive **delta-function potential** $\alpha\delta(x)$:

$$-\frac{\hbar^2}{2m}\psi''(x) + \alpha\,\delta(x)\,\psi(x) = E\,\psi(x)$$

**Physical context:** The delta potential is the simplest model of a short-range interaction. For $\alpha < 0$ (attractive), there is **exactly one bound state**: $\psi(x) = \sqrt{\kappa}\,e^{-\kappa|x|}$ with $E = -m\alpha^2/(2\hbar^2)$. The wavefunction has a characteristic cusp (discontinuous derivative) at $x=0$ due to the delta function.

In [ ]:
# Symbolic solve
delta_pot = registry.get('schrodinger_delta_potential')
sym_sol = delta_pot.symbolic_solve(hbar=1.0, mass=1.0, alpha=-1.0)
print(f'kappa = {sym_sol.info["kappa"]:.6f}')
print(f'Energy E = {sym_sol.info["E"]:.6f}')
print(f'Number of bound states: {sym_sol.info["n_bound_states"]}')
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and plot
num_sol = delta_pot.numerical_solve(
    t_span=(-8.0, 8.0), hbar=1.0, mass=1.0, alpha=-1.0,
)
x_num, y_num = num_sol.numerical
psi = y_num[0]

# Construct V(x) for visualization (approximate delta as narrow Gaussian)
V_display = np.zeros_like(x_num)
sigma_approx = 0.1
V_display = -1.0 / (sigma_approx * np.sqrt(2 * np.pi)) * np.exp(-x_num**2 / (2 * sigma_approx**2))

fig = plot_potential_and_wavefunction(
    x_num, V_display, [psi], energies=[num_sol.info['E']],
    title='Delta Potential: Single Bound State $\\psi = \\sqrt{\\kappa}\\,e^{-\\kappa|x|}$'
)
plt.show()

## 6. Time-Dependent Schrodinger Equation

The **time-dependent Schrodinger equation** for a free particle wave packet:

$$i\hbar\,\frac{\partial\psi}{\partial t} = -\frac{\hbar^2}{2m}\,\frac{\partial^2\psi}{\partial x^2}$$

**Physical context:** A Gaussian wave packet $\psi(x,0) = e^{-(x-x_0)^2/(4\sigma^2)} e^{ik_0 x}$ evolves in time, spreading due to dispersion while its centre moves at the group velocity $v_g = \hbar k_0/m$. The split-step Fourier method preserves unitarity exactly for the free-particle Hamiltonian.

In [ ]:
# Symbolic solve — analytic Gaussian wave packet
td_se = registry.get('time_dependent_schrodinger')
sym_sol = td_se.symbolic_solve(hbar=1.0, mass=1.0, sigma=0.1, k0=5.0)
print('Method:', sym_sol.info.get('method'))
print(f'Group velocity: {sym_sol.info["group_velocity"]:.4f}')
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve — split-step Fourier method
num_sol = td_se.numerical_solve(
    t_span=(0.0, 0.5), hbar=1.0, mass=1.0, sigma=0.1, k0=5.0,
    L=4.0, N=512, dt=0.001,
)
# numerical = (x, t_arr, psi_matrix)
x_arr, t_arr, psi_matrix = num_sol.numerical
print(f'Solver: {num_sol.info["solver"]}')
print(f'Time steps: {num_sol.info["n_steps"]}, Frames: {num_sol.info["n_frames"]}')

# Plot |psi|^2 as heatmap (wave packet evolution)
prob_density = np.abs(psi_matrix)**2

fig1 = plot_pde_heatmap(
    x_arr, t_arr, prob_density,
    title='Wave Packet Evolution: $|\\psi(x,t)|^2$ (Gaussian, $k_0=5$)'
)
plt.show()

# Snapshots of |psi|^2 at several times
n_frames = len(t_arr)
snap_indices = [0, n_frames//4, n_frames//2, 3*n_frames//4, -1]
fig2 = plot_pde_snapshots(
    x_arr, prob_density, snap_indices,
    title='Wave Packet: $|\\psi(x,t)|^2$ at Selected Times'
)
plt.show()

## 7. Hydrogen Radial Equation

The **radial Schrodinger equation** for the hydrogen-like atom:

$$-\frac{\hbar^2}{2m}\left[R'' + \frac{2}{r}R'\right] + \left[-\frac{e^2}{r} + \frac{\hbar^2\ell(\ell+1)}{2mr^2}\right]R = E\,R$$

**Physical context:** The radial part of the hydrogen atom wavefunction. Solutions involve **associated Laguerre polynomials** $L_{n-\ell-1}^{2\ell+1}(\rho)$ with $\rho = 2r/(na_0)$. Energy levels are $E_n = -me^4/(2\hbar^2 n^2)$, giving the familiar Balmer, Lyman, and Paschen spectral series. The Bohr radius $a_0 = \hbar^2/(me^2)$ sets the atomic scale.

In [ ]:
# Symbolic solve — radial wavefunctions for n=1,2,3
hydrogen = registry.get('hydrogen_radial')
for n, l in [(1, 0), (2, 0), (2, 1), (3, 0)]:
    sym_sol = hydrogen.symbolic_solve(n=n, l=l, hbar=1.0, mass=1.0, e=1.0)
    if sym_sol.info.get('reason'):
        print(f'n={n}, l={l}: {sym_sol.info["reason"]}')
    else:
        print(f'n={n}, l={l}: E_{n} = {sym_sol.info["E_n"]:.6f}, a_0 = {sym_sol.info["a0"]:.4f}')
        if sym_sol.latex:
            display(Math(sym_sol.latex))

In [ ]:
# Numerical solve and plot radial wavefunctions R(r)
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.tab10(np.linspace(0, 1, 4))

for i, (n, l) in enumerate([(1, 0), (2, 0), (2, 1), (3, 0)]):
    num_sol = hydrogen.numerical_solve(
        t_span=(0.01, 30.0), n=n, l=l, hbar=1.0, mass=1.0, e=1.0,
    )
    if num_sol.numerical is not None:
        r, y = num_sol.numerical
        R = y[0]  # R(r) = u(r)/r
        ax.plot(r, R, color=colors[i], linewidth=2,
                label=f'$R_{{{n}{l}}}(r)$, $E_{{{n}}}$={num_sol.info["E_n"]:.4f}')

ax.set_xlabel('r (units of $a_0$)')
ax.set_ylabel('$R_{nl}(r)$')
ax.set_title('Hydrogen Radial Wavefunctions $R_{nl}(r)$')
ax.set_xlim(0, 25)
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 8. Pauli Equation (Spin-1/2 Precession)

The **Pauli equation** for a spin-1/2 particle in a magnetic field:

$$i\hbar\,\frac{d\chi}{dt} = -\gamma\,(\mathbf{B}\cdot\boldsymbol{\sigma})\,\chi$$

**Physical context:** This describes **Larmor precession** of a spin-1/2 particle (e.g., electron, proton) in a uniform magnetic field $\mathbf{B}$. The 2-component spinor $\chi = (\chi_\uparrow, \chi_\downarrow)^T$ evolves as $\chi(t) = e^{-i\omega t \sigma_z/2}\chi(0)$ with **Larmor frequency** $\omega = \gamma B_z$. The spin-up and spin-down probabilities oscillate sinusoidally, which is the basis of NMR and ESR spectroscopy.

In [ ]:
# Symbolic solve — Larmor precession
pauli = registry.get('pauli_equation')
sym_sol = pauli.symbolic_solve(hbar=1.0, gamma=1.0, Bx=0.0, By=0.0, Bz=1.0)
print(f'Larmor frequency: omega = {sym_sol.info["omega"]:.4f}')
print(f'Precession period: T = {sym_sol.info["period"]:.4f}')
if sym_sol.latex:
    display(Math(sym_sol.latex))

In [ ]:
# Numerical solve — spin-up/spin-down probability oscillation
# Start with spin-up state; apply Bz field => Larmor precession
num_sol = pauli.numerical_solve(
    initial_conditions={'chi_up_re': 1.0, 'chi_up_im': 0.0,
                        'chi_down_re': 0.0, 'chi_down_im': 0.0},
    t_span=(0.0, 20.0),
    hbar=1.0, gamma=1.0, Bx=0.5, By=0.0, Bz=1.0,  # tilted field
)
t, y = num_sol.numerical

# y[0]=Re(chi_up), y[1]=Im(chi_up), y[2]=Re(chi_down), y[3]=Im(chi_down)
prob_up = y[0]**2 + y[1]**2     # |chi_up|^2
prob_down = y[2]**2 + y[3]**2   # |chi_down|^2
sigma_z = prob_up - prob_down   # <sigma_z>

omega = num_sol.info['omega']
period = num_sol.info['period']
print(f'Omega = {omega:.4f}, Period = {period:.4f}')

# Plot spin-up/down probabilities (Larmor precession)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(t, prob_up, 'b-', linewidth=2, label='$|\\chi_\\uparrow|^2$ (spin up)')
ax1.plot(t, prob_down, 'r-', linewidth=2, label='$|\\chi_\\downarrow|^2$ (spin down)')
ax1.set_ylabel('Probability')
ax1.set_title(f'Larmor Precession: $B_x={num_sol.info["Bx"]}$, $B_z={num_sol.info["Bz"]}$, $\\omega={omega:.2f}$')
ax1.legend()
ax1.grid(True)

# Plot <sigma_z> expectation value
ax2.plot(t, sigma_z, 'g-', linewidth=2, label='$\\langle \\sigma_z \\rangle$')
ax2.axhline(y=0, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Time')
ax2.set_ylabel('$\\langle \\sigma_z \\rangle$')
ax2.set_title('Spin Polarization (Larmor Precession)')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()